# HybridGaussianConditional

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/hybrid/doc/HybridGaussianConditional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass  # Not running on Colab, do nothing

A `HybridGaussianConditional` represents a conditional probability distribution $P(X | M, Z)$ where the frontal variables $X$ are continuous, and the parent variables include both discrete variables $M$ and potentially other continuous variables $Z$. It inherits from `HybridGaussianFactor` and `Conditional<HybridGaussianFactor, HybridGaussianConditional>`.

Essentially, it's a collection of `GaussianConditional`s, where the choice of which `GaussianConditional` applies depends on the assignment $m$ of the discrete parent variables $M$.
$$
P(X | M=m, Z) = P_m(X | Z)
$$
where $P_m(X | Z)$ is a standard `GaussianConditional`.

Internally, it uses a `DecisionTree<Key, GaussianConditional::shared_ptr>` to store the different conditional components, indexed by the discrete parent `DiscreteKey`s $M$. Note that unlike `HybridGaussianFactor`, the scalar value stored alongside the factor in the base class represents the *negative log normalization constant* for that branch, ensuring each $P_m(X|Z)$ integrates to 1.

These conditionals are the standard result of eliminating continuous variables when discrete variables are present in the separator during hybrid elimination.

In [4]:
import gtsam
import numpy as np

from gtsam import (
    HybridGaussianConditional, GaussianConditional,
    DiscreteKeys, DiscreteValues, VectorValues, HybridValues,
    JacobianFactor
)
from gtsam.symbol_shorthand import X, D

## Initialization

Can be initialized with one or more discrete parent keys and the corresponding `GaussianConditional` components.

In [ ]:
# --- Example 1: Single Discrete Parent P(X0 | D0) ---
dk0 = (D(0), 2) # Binary discrete parent D0

# Gaussian conditional for mode D0=0: P(X0|D0=0) = N(0, 1)
# R=1, d=0
gc0 = GaussianConditional(X(0), np.zeros(1), np.eye(1), gtsam.noiseModel.Unit.Create(1))

# Gaussian conditional for mode D0=1: P(X0|D0=1) = N(5, 4=2^2)
# R=1/2=0.5, d = R*mean = 0.5*5 = 2.5
# Need noise model with sigma=2.0
noise1 = gtsam.noiseModel.Isotropic.Sigma(1, 2.0)
gc1 = GaussianConditional(X(0), np.array([2.5]), np.eye(1)*0.5, noise1)

hgc1 = HybridGaussianConditional(dk0, [gc0, gc1])
print("HybridGaussianConditional P(X0 | D0):")
hgc1.print()

# --- Example 2: Multiple Discrete Parents + Continuous Parent P(X1 | D0, D1, X0) ---
dk1 = (D(1), 2)
discrete_parents = DiscreteKeys()
discrete_parents.push_back(dk0)
discrete_parents.push_back(dk1)
cont_parent = X(0)

# Define GaussianConditionals P(X1 | X0) for each discrete mode (D0, D1)
# Mode (0,0): N(X1; X0 + 0, 1) -> R=1, S=-1, d=0
gc_00 = GaussianConditional(X(1), np.zeros(1), np.eye(1), cont_parent, -np.eye(1), gtsam.noiseModel.Unit.Create(1))
# Mode (1,0): N(X1; X0 + 1, 1) -> R=1, S=-1, d=1
gc_10 = GaussianConditional(X(1), np.ones(1), np.eye(1), cont_parent, -np.eye(1), gtsam.noiseModel.Unit.Create(1))
# Mode (0,1): N(X1; X0 + 2, 1) -> R=1, S=-1, d=2
gc_01 = GaussianConditional(X(1), np.ones(1)*2, np.eye(1), cont_parent, -np.eye(1), gtsam.noiseModel.Unit.Create(1))
# Mode (1,1): N(X1; X0 + 3, 1) -> R=1, S=-1, d=3
gc_11 = GaussianConditional(X(1), np.ones(1)*3, np.eye(1), cont_parent, -np.eye(1), gtsam.noiseModel.Unit.Create(1))

# Need to build a DecisionTree<Key, GaussianConditional::shared_ptr>
# Assume order D0, D1 -> (0,0), (1,0), (0,1), (1,1) based on key order
try:
    cond_tree = gtsam.DecisionTreeGaussianConditional() # Assumed class/constructor
    cond_tree.add(discrete_parents, [gc_00, gc_10, gc_01, gc_11])
    hgc2 = HybridGaussianConditional(discrete_parents, cond_tree)
    print("\nHybridGaussianConditional P(X1 | D0, D1, X0):")
    hgc2.print()
except Exception as e:
    print("\nMulti-key HybridGaussianConditional construction might require specific DecisionTree setup.")
    print(f"Error: {e}")


# --- Example 3: Constructor with Parameters (P(X0 | D0)) ---
# parameters = vector<pair<Vector, double>> (mean, sigma)
params = [(np.array([0.0]), 1.0), (np.array([5.0]), 2.0)]
hgc3 = HybridGaussianConditional(dk0, X(0), params)
print("\nHybridGaussianConditional P(X0 | D0) from params:")
hgc3.print() # Should match hgc1

HybridGaussianConditional P(X0 | D0):
HybridGaussianConditional

 P( x0 | d0)
 Discrete Keys = (d0, 2), 
 logNormalizationConstant: -0.918939

 Choice(d0) 
 0 Leaf p(x0)
  R = [ 1 ]
  d = [ 0 ]
  mean: 1 elements
  x0: 0
  logNormalizationConstant: -0.918939
  Noise model: unit (1) 

 1 Leaf p(x0)
  R = [ 0.5 ]
  d = [ 2.5 ]
  mean: 1 elements
  x0: 5
  logNormalizationConstant: -2.30523
isotropic dim=1 sigma=2


Multi-key HybridGaussianConditional construction might require specific DecisionTree setup.
Error: module 'gtsam' has no attribute 'DecisionTreeGaussianConditional'


TypeError: __init__(): incompatible constructor arguments. The following argument types are supported:
    1. gtsam.gtsam.HybridGaussianConditional(discreteParents: gtsam.gtsam.DiscreteKeys, conditionals: gtsam::DecisionTree<unsigned __int64,std::shared_ptr<gtsam::GaussianConditional> >)
    2. gtsam.gtsam.HybridGaussianConditional(discreteParent: tuple[int, int], conditionals: list[gtsam.gtsam.GaussianConditional])

Invoked with: (7205759403792793600, 2), 8646911284551352320, [(array([0.]), 1.0), (array([5.]), 2.0)]

Did you forget to `#include <pybind11/stl.h>`? Or <pybind11/complex.h>,
<pybind11/functional.h>, <pybind11/chrono.h>, etc. Some automatic
conversions are optional and require extra headers to be included
when compiling your pybind11 module.

## Accessing Components and Properties

You can retrieve the underlying decision tree of conditionals and select a specific `GaussianConditional` based on a discrete assignment.

In [6]:
# Using hgc1 from Example 1: P(X0 | D0) with modes N(0,1) and N(5,4)
print("Accessing components of hgc1:")

# Get the decision tree of GaussianConditionals
conditional_tree = hgc1.conditionals() # Note: This might involve dynamic casts
print("Internal Conditional Tree:")
conditional_tree.print()

# Get the GaussianConditional for a specific assignment using choose() or ()
assignment0 = gtsam.DiscreteValues([(D(0), 0)])
selected_gc0 = hgc1.choose(assignment0) # or hgc1(assignment0)
print("\nSelected GaussianConditional for D0=0:")
selected_gc0.print() # Should be N(0, 1)

assignment1 = gtsam.DiscreteValues([(D(0), 1)])
selected_gc1 = hgc1(assignment1)
print("\nSelected GaussianConditional for D0=1:")
selected_gc1.print() # Should be N(5, 4)

# Access conditional properties
print(f"\nFrontals: {hgc1.frontals()}")
print(f"Parents: {hgc1.parents()}")
print(f"Discrete Parents (part of HybridFactor): {hgc1.discreteKeys()}")
print(f"Continuous Parents: {hgc1.continuousParents()}") # Convenience method

Accessing components of hgc1:


AttributeError: 'gtsam.gtsam.HybridGaussianConditional' object has no attribute 'conditionals'

## Evaluation (`logProbability`, `evaluate`)

These methods evaluate the probability density $P_m(X | Z)$ corresponding to the selected discrete mode $m$. They require a `HybridValues` object containing assignments for all involved variables (frontal $X$, discrete parents $M$, continuous parents $Z$).

In [ ]:
# Using hgc1: P(X0 | D0) with modes N(0,1) and N(5,4)

# --- Evaluate mode 0: P(X0=0.1 | D0=0) ---
hybrid_vals0 = gtsam.HybridValues()
hybrid_vals0.insert(X(0), np.array([0.1])) # Frontal value
hybrid_vals0.insert(D(0), 0)             # Discrete parent assignment

log_prob0 = hgc1.logProbability(hybrid_vals0)
prob0 = hgc1.evaluate(hybrid_vals0)
# Expected: log density of N(0.1; 0, 1)
print(f"\nP(X0=0.1 | D0=0): LogProb={log_prob0}, Prob={prob0}")

# --- Evaluate mode 1: P(X0=4.0 | D0=1) ---
hybrid_vals1 = gtsam.HybridValues()
hybrid_vals1.insert(X(0), np.array([4.0])) # Frontal value
hybrid_vals1.insert(D(0), 1)             # Discrete parent assignment

log_prob1 = hgc1.logProbability(hybrid_vals1)
prob1 = hgc1.evaluate(hybrid_vals1)
# Expected: log density of N(4.0; 5, 4)
print(f"P(X0=4.0 | D0=1): LogProb={log_prob1}, Prob={prob1}")

# Note: The error() method inherited from HybridGaussianFactor gives
# the Gaussian negative log-likelihood *plus* the neg log normalization constant
# stored in the tree.
# error = -log P(x|m,z) - log(NormConstant)
err0 = hgc1.error(hybrid_vals0)
print(f"Error(X0=0.1|D0=0): {err0} (Should be approx -log_prob0 - negLogConstant(gc0))")
neg_log_const_0 = -gc0.logNormalizationConstant() # Should be ~0.9189
print(f"  Check: {-log_prob0 + neg_log_const_0}")

## Restriction (`restrict`)

Fixes the discrete parent variables according to a given assignment, returning the corresponding `GaussianConditional` component (as a `gtsam.Factor::shared_ptr`, which needs casting).

In [7]:
# Using hgc1
assignment = gtsam.DiscreteValues([(D(0), 1)]) # Choose mode 1
restricted_factor_ptr = hgc1.restrict(assignment)

if restricted_factor_ptr:
    restricted_factor_ptr.print("\nRestricted Factor for D0=1:")
    # Should be equivalent to gc1: P(X0|D0=1) = N(5, 4)
    # Try casting to verify
    restricted_cond = gtsam.GaussianConditional.dynamic_cast(restricted_factor_ptr)
    if restricted_cond:
         print("Successfully cast restricted factor to GaussianConditional.")
    else:
         print("Failed to cast restricted factor to GaussianConditional (might be Factor base).")
else:
    print("\nRestriction failed.")

TypeError: __init__(): incompatible constructor arguments. The following argument types are supported:
    1. gtsam.gtsam.DiscreteValues()

Invoked with: [(7205759403792793600, 1)]